# Финальный pipeline — часть 1: сбор pool и ручная разметка

Здесь:
1. загружаем запросы и модели (bi- и cross-encoder),
2. для каждого запроса собираем pool = union(top-20 bi-encoder, top-20 после rerank),
3. размечаем пары вручную (0/1/2) через интерактивный виджет.

Метрики считаются в отдельном ноутбуке `pipeline-50k-major_metrics-evaluation.ipynb` — после того как разметка `ground_truth_pool.json` готова.


In [1]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали из подпапки
import os
from pathlib import Path
_p = Path.cwd()
while _p.name != 'thesis' and _p.parent != _p:
    _p = _p.parent
if _p.name == 'thesis':
    os.chdir(_p)
print('CWD:', Path.cwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import math
import time
from pathlib import Path
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd
import torch
import lancedb
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')


DEVICE: cpu


In [3]:
# ==================== КОНФИГУРАЦИЯ ====================

# Выбранный bi-encoder для финального пайплайна (лучший по Spearman на golden_eval)
BI_ENCODER_PATH   = 'models/bi-encoder-e5-finetuned'
BI_ENCODER_TABLE  = 'e5-base-fine-tuned-50k'
BI_DOC_PREFIX     = 'passage: '   # E5 требует префиксы
BI_QUERY_PREFIX   = 'query: '

CROSS_ENCODER_PATH = 'models/final/cross-encoder'

LANCEDB_PATH  = './lancedb_store'

# Ground truth pool (собирается один раз, размечается вручную)
GT_PAIRS_JSON       = 'ground_truth_pairs.json'          # исходные описания товаров
POOL_CANDIDATES_JSON = 'benchmark/pipeline/pool_candidates.json'  # сам pool (генерируется)
GT_POOL_JSON        = 'benchmark/pipeline/ground_truth_pool.json' # разметка (заполняется руками)

# Индексы описаний из ground_truth_pairs.json, которые НЕ используем
# (#6 — дубль #5 жиросжигатель; #14 — дубль #13 эзотерика)
DROP_QUERY_INDEXES = {6, 14}

# Сколько постов берём в pool от каждого режима
TOP_K_BI     = 20    # top-K retrieve-only
TOP_K_RERANK = 20    # top-K после cross-encoder (pool = union этих двух)

# Для cross-encoder reranking'а bi-encoder отдаёт больше кандидатов
TOP_K_BI_FOR_RERANK = 100

# Метрики считаются @10
K_METRIC = 10


## 1. Запросы

17 описаний товаров (из 19 исходных убраны дубли #6 «жиросжигатель» и #14 «скретч-открытки»).

In [4]:
# Загружаем описания товаров, исключаем дубли
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    all_pairs = json.load(f)

queries = []
for idx, pair in enumerate(all_pairs, 1):
    if idx in DROP_QUERY_INDEXES:
        continue
    queries.append({
        'query_idx':    idx,
        'imt_name':     pair.get('imt_name', ''),
        'description':  pair['description'],
    })

print(f'Запросов для пайплайна: {len(queries)} (из {len(all_pairs)}, убраны: {sorted(DROP_QUERY_INDEXES)})')
for q in queries:
    print(f"  {q['query_idx']:2d}. {q['imt_name']}")


Запросов для пайплайна: 17 (из 19, убраны: [6, 14])
   1. Затирка для плитки готовая - белая
   2. Самоклеящиеся панели для стен на кухню 60х30см пвх 15шт
   3. Развивашки 2-3-4 года/пиши стирай тетрадь/книги для малышей
   4. Детская мозаика (5 цветов, 40 элементов) "Кораблик"
   5. Жиросжигатель для похудения женщинам 60 капсул
   7. Накидка на сиденье DongFeng Fengshen Yixuan GS
   8. Кроссовер Monjaro
   9. Матрас надувной двуспальный 203х152см с подушками и насосом
  10. Гуд Найт Мягкое фито снотворное для сна
  11. Клавиатура игровая с подсветкой

  12. Видеокарта RTX 3050 6 ГБ RTL (RTX 3050 LP E 6G OC)
  13. Карты таро уэйта для начинающих с инструкцией обучающие
  15. Подушка для путешествий Travel Blue Tranquility Pillow (212)
  16. Чехол на чемодан L плотный на молнии с рисунком
  17. Спиннинг на щуку для рыбалки КATANA 2,1 м 5-25 г 15-40 г
  18. Кормушка для рыбалки Флэт - монтаж карповый фидерный
  19. Шляпа с декоративной цепочкой



## 2. Загрузка моделей

In [5]:
# Загружаем bi-encoder и cross-encoder
print(f'Загрузка bi-encoder: {BI_ENCODER_PATH}')
bi_encoder = SentenceTransformer(BI_ENCODER_PATH, device=DEVICE)
print(f'  dim={bi_encoder.get_sentence_embedding_dimension()}')

print(f'Загрузка cross-encoder: {CROSS_ENCODER_PATH}')
cross_encoder = CrossEncoder(CROSS_ENCODER_PATH, device=DEVICE)
print('  готово')

# LanceDB
db = lancedb.connect(LANCEDB_PATH)
assert BI_ENCODER_TABLE in db.table_names(), f'Нет таблицы {BI_ENCODER_TABLE}'
table = db.open_table(BI_ENCODER_TABLE)
print(f'Таблица {BI_ENCODER_TABLE}: {table.count_rows():,} постов')


You are trying to use a model that was created with Sentence Transformers version 5.3.0, but you're currently using version 5.2.3. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


Загрузка bi-encoder: models/bi-encoder-e5-finetuned
  dim=768
Загрузка cross-encoder: models/final/cross-encoder


The tokenizer you are loading from 'models/final/cross-encoder' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  готово
Таблица e5-base-fine-tuned-50k: 50,000 постов


In [6]:
def encode_query(q):
    q_in = (BI_QUERY_PREFIX + q) if BI_QUERY_PREFIX else q
    return bi_encoder.encode([q_in], normalize_embeddings=True)[0].tolist()


def retrieve(query, k):
    qvec = encode_query(query)
    return (table.search(qvec, query_type='vector')
                 .limit(k)
                 .select(['text', 'channel', 'category'])
                 .to_list())


def rerank(query, candidates):
    pairs = [[query, c['text']] for c in candidates]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores))
    reranked = [candidates[i] for i in order]
    return reranked, [float(scores[i]) for i in order]


## 3. Сбор pool кандидатов

Для каждого запроса собираем объединение top-20 от bi-encoder (retrieve-only) и top-20 после cross-encoder reranking top-100. Файл `pool_candidates.json` генерируется один раз и потом переиспользуется.

In [7]:
# ============================================================
# Сбор pool кандидатов (делается ОДИН РАЗ)
# ============================================================
# Для каждого запроса:
#   1) bi-encoder → top-20 (retrieve-only pool)
#   2) bi-encoder → top-100 → cross-encoder → top-20 (rerank pool)
#   3) Объединяем оба, дедуплицируем по тексту поста
#
# Если файл POOL_CANDIDATES_JSON уже существует — пропускаем сбор.

def collect_pool(force=False):
    if os.path.exists(POOL_CANDIDATES_JSON) and not force:
        with open(POOL_CANDIDATES_JSON, encoding='utf-8') as f:
            existing = json.load(f)
        print(f'Pool уже собран: {POOL_CANDIDATES_JSON}')
        total = sum(len(p['candidates']) for p in existing)
        print(f'  запросов: {len(existing)}, всего пар для разметки: {total}')
        return existing

    pool = []
    for q in queries:
        # Retrieve-only top-20
        ret_only = retrieve(q['description'], TOP_K_BI)

        # Retrieve→rerank: берём top-100 от bi-encoder, ре-ранжируем, оставляем top-20
        ret_for_rerank = retrieve(q['description'], TOP_K_BI_FOR_RERANK)
        reranked, _ = rerank(q['description'], ret_for_rerank)
        top_rerank = reranked[:TOP_K_RERANK]

        # Объединяем по тексту (текст — уникальный ключ, т.к. в таблице он уникален)
        seen = set()
        merged = []
        for src_name, rows in [('bi', ret_only), ('rerank', top_rerank)]:
            for r in rows:
                key = r['text'].strip()
                if key in seen:
                    continue
                seen.add(key)
                merged.append({
                    'text':     r['text'],
                    'channel':  r['channel'],
                    'category': r.get('category', ''),
                    'source':   [src_name],  # заполним ниже
                })
        # помечаем, из какого режима (или обоих) пришёл пост
        texts_in_ret = {r['text'].strip() for r in ret_only}
        texts_in_rer = {r['text'].strip() for r in top_rerank}
        for m in merged:
            srcs = []
            if m['text'].strip() in texts_in_ret: srcs.append('bi')
            if m['text'].strip() in texts_in_rer: srcs.append('rerank')
            m['source'] = srcs

        pool.append({
            'query_idx':    q['query_idx'],
            'imt_name':     q['imt_name'],
            'description':  q['description'],
            'candidates':   merged,
        })
        print(f"  запрос {q['query_idx']:2d} ({q['imt_name'][:50]}): {len(merged)} уникальных кандидатов")

    os.makedirs(os.path.dirname(POOL_CANDIDATES_JSON), exist_ok=True)
    with open(POOL_CANDIDATES_JSON, 'w', encoding='utf-8') as f:
        json.dump(pool, f, ensure_ascii=False, indent=2)

    total = sum(len(p['candidates']) for p in pool)
    print(f'\nPool собран: {len(pool)} запросов, всего {total} пар. Сохранено в {POOL_CANDIDATES_JSON}')
    return pool

pool = collect_pool(force=False)


Pool уже собран: benchmark/pipeline/pool_candidates.json
  запросов: 17, всего пар для разметки: 814


## 4. Ручная разметка pool

Интерактивная ячейка: показывает описание товара и пост рядом (полный текст без обрезаний), под ними — кнопки **0 / 1 / 2**, плюс «Назад», «Пропустить», «Выход».

**Автосохранение после каждого ответа** в `ground_truth_pool.json` — можно прерваться в любой момент и продолжить с того же места при следующем запуске ячейки.

Шкала:
- **0** — не релевантно (разные темы)
- **1** — частично релевантно (смежная тема, есть связь)
- **2** — полностью релевантно (точное попадание, товар рекламируется)

In [ ]:
# ============================================================
# ИНТЕРАКТИВНАЯ РАЗМЕТКА POOL
# ============================================================
# Показываем полный текст описания товара и полный текст поста рядом,
# без обрезаний. Кнопки: 0 / 1 / 2 / Назад / Выход.
# Автосохранение в GT_POOL_JSON ПОСЛЕ КАЖДОГО ОТВЕТА — резюмируется с места.
#
# Шкала:
#   0 — не релевантно (разные темы)
#   1 — частично релевантно (смежная тема, есть связь)
#   2 — полностью релевантно (точное попадание / товар рекламируется)
# ============================================================

# Плоский список пар (query_idx, cand_idx, query_text, post_text, meta)
flat = []
for p in pool:
    for ci, c in enumerate(p['candidates']):
        flat.append({
            'query_idx':   p['query_idx'],
            'cand_idx':    ci,
            'imt_name':    p['imt_name'],
            'description': p['description'],
            'post_text':   c['text'],
            'channel':     c['channel'],
            'category':    c['category'],
            'source':      c['source'],
        })
print(f'Всего пар в pool: {len(flat)}')

# Загружаем существующую разметку (если есть)
def load_annotations():
    if os.path.exists(GT_POOL_JSON):
        with open(GT_POOL_JSON, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_annotations(ann):
    os.makedirs(os.path.dirname(GT_POOL_JSON), exist_ok=True)
    with open(GT_POOL_JSON, 'w', encoding='utf-8') as f:
        json.dump(ann, f, ensure_ascii=False, indent=2)

annotations = load_annotations()
# ключ: f"{query_idx}:{cand_idx}"  →  score int 0/1/2
print(f'Уже размечено: {len(annotations)} / {len(flat)}')


def ann_key(item):
    return f"{item['query_idx']}:{item['cand_idx']}"


def render_pair_html(item, progress_done, progress_total):
    src_badges = ''.join(
        f'<span style="background:#495057;color:#fff;padding:2px 8px;border-radius:3px;'
        f'margin-right:4px;font-size:11px;">{s}</span>' for s in item['source']
    )
    return f'''
    <div style="font-family:system-ui,sans-serif;color:#000;max-width:1400px;">
        <div style="display:flex;gap:12px;align-items:center;margin-bottom:12px;">
            <div style="font-size:16px;font-weight:bold;">
                Запрос {item['query_idx']} · кандидат {item['cand_idx']+1}
            </div>
            <div style="font-size:13px;color:#495057;">
                Прогресс: {progress_done} / {progress_total}
            </div>
            <div>{src_badges}</div>
        </div>
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;">
            <div style="background:#e7f3ff;border:2px solid #cfe2ff;border-radius:8px;padding:16px;">
                <div style="font-weight:bold;font-size:12px;text-transform:uppercase;
                            margin-bottom:8px;letter-spacing:0.5px;color:#1864ab;">
                    Описание товара · {item['imt_name']}
                </div>
                <div style="font-size:14px;line-height:1.5;white-space:pre-wrap;">{item['description']}</div>
            </div>
            <div style="background:#fff9e6;border:2px solid #ffe69c;border-radius:8px;padding:16px;">
                <div style="font-weight:bold;font-size:12px;text-transform:uppercase;
                            margin-bottom:8px;letter-spacing:0.5px;color:#a6791c;">
                    Пост · @{item['channel']} · {item['category']}
                </div>
                <div style="font-size:14px;line-height:1.5;white-space:pre-wrap;">{item['post_text']}</div>
            </div>
        </div>
    </div>
    '''


def find_next_unlabeled(start_idx):
    for i in range(start_idx, len(flat)):
        if ann_key(flat[i]) not in annotations:
            return i
    return None


# Состояние: индекс в flat[]
current_idx = [find_next_unlabeled(0)]

output = widgets.Output()

btn_0 = widgets.Button(description='0 · не релевантно', button_style='danger',
                       layout=widgets.Layout(width='180px', height='45px'))
btn_1 = widgets.Button(description='1 · частично',       button_style='warning',
                       layout=widgets.Layout(width='180px', height='45px'))
btn_2 = widgets.Button(description='2 · полностью',       button_style='success',
                       layout=widgets.Layout(width='180px', height='45px'))
btn_back = widgets.Button(description='← Назад',    layout=widgets.Layout(width='120px', height='45px'))
btn_skip = widgets.Button(description='Пропустить →', layout=widgets.Layout(width='140px', height='45px'))
btn_exit = widgets.Button(description='✕ Выход',    layout=widgets.Layout(width='120px', height='45px'))

buttons = widgets.HBox([btn_0, btn_1, btn_2,
                        widgets.HTML('<div style="width:20px"></div>'),
                        btn_back, btn_skip, btn_exit])


def show_current():
    output.clear_output(wait=True)
    idx = current_idx[0]
    with output:
        if idx is None:
            display(HTML('<h2 style="color:#28a745;">✓ Все пары размечены</h2>'))
            return
        item = flat[idx]
        done = sum(1 for i in range(len(flat)) if ann_key(flat[i]) in annotations)
        display(HTML(render_pair_html(item, done, len(flat))))


def on_score(score):
    def handler(_):
        idx = current_idx[0]
        if idx is None:
            return
        item = flat[idx]
        annotations[ann_key(item)] = score
        save_annotations(annotations)
        current_idx[0] = find_next_unlabeled(idx + 1)
        show_current()
    return handler


def on_back(_):
    idx = current_idx[0]
    # идём назад до первой размеченной, снимаем её и показываем
    start = (idx - 1) if idx is not None else (len(flat) - 1)
    for i in range(start, -1, -1):
        if ann_key(flat[i]) in annotations:
            del annotations[ann_key(flat[i])]
            save_annotations(annotations)
            current_idx[0] = i
            show_current()
            return
    # нечего откатывать
    show_current()


def on_skip(_):
    idx = current_idx[0]
    if idx is None:
        return
    current_idx[0] = find_next_unlabeled(idx + 1)
    show_current()


def on_exit(_):
    output.clear_output(wait=True)
    with output:
        done = len(annotations)
        display(HTML(f'<h3>Выход. Размечено: {done} / {len(flat)}. Данные сохранены в {GT_POOL_JSON}.</h3>'))


btn_0.on_click(on_score(0))
btn_1.on_click(on_score(1))
btn_2.on_click(on_score(2))
btn_back.on_click(on_back)
btn_skip.on_click(on_skip)
btn_exit.on_click(on_exit)

display(buttons, output)
show_current()


Всего пар в pool: 814
Уже размечено: 457 / 814


Output()